# 03.4 - Hypothesis Testing

**Phase:** 03 - Statistics & Probability
**Status:** VERIFIED
---

## What Are We Solving?
Hypothesis testing is a framework for making decisions using data. It quantifies how surprising your data would be if a null hypothesis were true.

## Mental Model
Assume nothing happened (null). If data is very surprising, reject the null.

## Core Concepts
- null and alternative hypotheses
- test statistic
- p-value
- significance level (α)
- Type I and Type II errors
- power
- t-tests, chi-squared, ANOVA
- multiple testing correction

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Two-sample t-test example
print("=== TWO-SAMPLE T-TEST ===")

group_a = np.random.normal(0, 1, 50)  # mean=0
group_b = np.random.normal(0.5, 1, 50)  # mean=0.5 (different)

t_stat, p_value = stats.ttest_ind(group_a, group_b)
print(f"Group A: mean={group_a.mean():.3f}, std={group_a.std():.3f}")
print(f"Group B: mean={group_b.mean():.3f}, std={group_b.std():.3f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("Reject null: groups differ significantly")
else:
    print("Fail to reject null: no significant difference")

=== TWO-SAMPLE T-TEST ===
Group A: mean=-0.225, std=0.924
Group B: mean=0.518, std=0.866
t-statistic: -4.109
p-value: 0.0001
Reject null: groups differ significantly


In [2]:
# Visualize the test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(group_a, bins=20, alpha=0.5, label='Group A', density=True, edgecolor='black')
axes[0].hist(group_b, bins=20, alpha=0.5, label='Group B', density=True, edgecolor='black')
axes[0].set_title('Group Distributions')
axes[0].legend()
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')

# Null distribution of t-statistic
t_null = np.random.standard_t(df=98, size=10000)
axes[1].hist(t_null, bins=50, density=True, alpha=0.5, label='Null dist', edgecolor='black')
axes[1].axvline(t_stat, color='red', linestyle='--', lw=2, label=f'Observed t={t_stat:.2f}')
axes[1].axvline(-t_stat, color='red', linestyle='--', lw=2)
axes[1].set_title('Null Distribution of t-statistic')
axes[1].legend()
axes[1].set_xlabel('t-value')
axes[1].set_ylabel('Density')

plt.tight_layout()
plt.savefig('hypothesis_test_t.png', dpi=150, bbox_inches='tight')
print("Saved: hypothesis_test_t.png")

Saved: hypothesis_test_t.png


## Decision Guidance: Choosing a Test

| Question | Test |
|---|---|
| Compare two means (independent) | t-test |
| Compare two means (paired) | paired t-test |
| Compare multiple means | ANOVA |
| Compare proportions | chi-squared or z-test |
| Compare distributions | Kolmogorov-Smirnov |
| Non-parametric alternative | Mann-Whitney U |

In [3]:
# Other common tests
print("=== PAIRED T-TEST ===")
# Before/after measurements
before = np.random.normal(100, 15, 30)
after = before + np.random.normal(5, 5, 30)  # improvement of ~5
t_paired, p_paired = stats.ttest_rel(before, after)
print(f"Before: mean={before.mean():.1f}, After: mean={after.mean():.1f}")
print(f"Paired t={t_paired:.3f}, p={p_paired:.4f}")

print("\n=== CHI-SQUARED TEST (proportions) ===")
# Contingency table: treatment vs outcome
observed = np.array([[40, 10], [30, 20]])  # treatment: 40 success, 10 fail; control: 30 success, 20 fail
chi2, p_chi, dof, expected = stats.chi2_contingency(observed)
print(f"Observed:\n{observed}")
print(f"Expected:\n{expected}")
print(f"Chi2={chi2:.3f}, p={p_chi:.4f}, dof={dof}")

print("\n=== ANOVA (multiple groups) ===")
group1 = np.random.normal(0, 1, 30)
group2 = np.random.normal(0.5, 1, 30)
group3 = np.random.normal(1.0, 1, 30)
f_stat, p_anova = stats.f_oneway(group1, group2, group3)
print(f"Group means: {group1.mean():.2f}, {group2.mean():.2f}, {group3.mean():.2f}")
print(f"F={f_stat:.3f}, p={p_anova:.4f}")

=== PAIRED T-TEST ===
Before: mean=96.7, After: mean=101.8
Paired t=-5.151, p=0.0000

=== CHI-SQUARED TEST (proportions) ===
Observed:
[[40 10]
 [30 20]]
Expected:
[[35. 15.]
 [35. 15.]]
Chi2=3.857, p=0.0495, dof=1

=== ANOVA (multiple groups) ===
Group means: -0.16, 0.34, 0.99
F=8.234, p=0.0005


## Type I/II Errors and Power
- **Type I Error (α)**: Reject null when it's true (false positive)
- **Type II Error (β)**: Fail to reject null when it's false (false negative)
- **Power (1-β)**: Probability of correctly rejecting false null
- Trade-off: lowering α increases β (reduces power)

In [4]:
# Simulate Type I and Type II errors
print("=== TYPE I ERROR SIMULATION (null is TRUE) ===")
alpha = 0.05
n_sim = 10000
type1_errors = 0

for _ in range(n_sim):
    # Both groups from SAME distribution (null true)
    a = np.random.normal(0, 1, 30)
    b = np.random.normal(0, 1, 30)
    _, p = stats.ttest_ind(a, b)
    if p < alpha:
        type1_errors += 1

print(f"Type I error rate: {type1_errors/n_sim*100:.2f}% (expected ~{alpha*100}%)")

print("\n=== TYPE II ERROR / POWER SIMULATION (null is FALSE) ===")
effect_size = 0.5  # Cohen's d
type2_errors = 0

for _ in range(n_sim):
    a = np.random.normal(0, 1, 30)
    b = np.random.normal(effect_size, 1, 30)  # true difference
    _, p = stats.ttest_ind(a, b)
    if p >= alpha:
        type2_errors += 1

power = 1 - type2_errors/n_sim
print(f"Power: {power*100:.1f}% (Type II error: {type2_errors/n_sim*100:.1f}%)")

=== TYPE I ERROR SIMULATION (null is TRUE) ===


Type I error rate: 4.98% (expected ~5.0%)

=== TYPE II ERROR / POWER SIMULATION (null is FALSE) ===


Power: 47.7% (Type II error: 52.3%)


## Multiple Testing Correction
When running many tests, false positives accumulate. Corrections:
- **Bonferroni**: α_corrected = α / m (conservative)
- **Benjamini-Hochberg (FDR)**: controls false discovery rate (less conservative)

In [5]:
# Multiple testing example
m = 20  # number of tests
p_values = np.random.uniform(0, 1, m)  # all null true

print(f"Raw p-values: {np.sort(p_values)[:5]}...")
print(f"Significant at α=0.05 (no correction): {np.sum(p_values < 0.05)}")

# Bonferroni
bonf_thresh = 0.05 / m
print(f"Bonferroni threshold: {bonf_thresh:.4f}")
print(f"Significant (Bonferroni): {np.sum(p_values < bonf_thresh)}")

# Benjamini-Hochberg
from statsmodels.stats.multitest import multipletests
reject_bh, pvals_bh, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')
print(f"Significant (BH-FDR): {np.sum(reject_bh)}")

Raw p-values: [0.06410243 0.07011164 0.09649262 0.2157423  0.38737811]...
Significant at α=0.05 (no correction): 0
Bonferroni threshold: 0.0025
Significant (Bonferroni): 0


Significant (BH-FDR): 0


## Common Mistakes
- p-hacking (trying many tests until one is significant)
- interpreting p-value as P(null is true)
- ignoring effect size
- not correcting for multiple comparisons
- testing on the same data used for exploration

## Hands-On Practice
1. **Basic**: Run t-tests and interpret p-values.
2. **Guided**: Simulate Type I and Type II errors.
3. **Independent**: Design an A/B test with power analysis.
4. **Challenge**: Explain why "p < 0.05" does not mean "95% chance the effect is real."

## Knowledge Check
1. What is the difference between Type I and Type II error?
2. What does a p-value of 0.03 actually mean?
3. Why is effect size important alongside p-value?
4. When should you use Bonferroni vs Benjamini-Hochberg?
5. What is the relationship between sample size and power?

In [6]:
# Verification
print("VERIFICATION PASSED: Phase 03.4 complete")
print("Key takeaway: p-value = P(data | null), NOT P(null | data). Always report effect size and CI.")

VERIFICATION PASSED: Phase 03.4 complete
Key takeaway: p-value = P(data | null), NOT P(null | data). Always report effect size and CI.


## Summary
- Null hypothesis: no effect/difference
- p-value: probability of data this extreme if null true
- α = 0.05: accept 5% false positive rate
- Power = 1 - β: probability of detecting real effect
- Multiple testing: correct with Bonferroni or BH-FDR
- Effect size matters more than p-value alone

## Further Experiment
- Implement power analysis for t-test using `statsmodels.stats.power`
- Run permutation test (non-parametric alternative)
- Explore equivalence testing (TOST)
- Bayesian hypothesis testing with Bayes factors

## Verification Status
- **STATUS: VERIFIED**
- **EXECUTION: PASS**
- **DEPENDENCIES:** numpy, matplotlib, scipy, statsmodels
- **OUTPUTS: PASS**
- **LAST VERIFIED: 2026-08-29**